# Explainable RAG for Clinical Question Answering
### Colab  implementation — PubMedQA

**Author:** Zaid Ahmed A Zardi
**Pipeline:** BioBERT embeddings → FAISS retrieval → quantized LLM generation → explainability layer → evaluation

This notebook is built to run end-to-end on a ** Colab T4 GPU**. It:
1. Builds a retrieval corpus from a **subsample of PQA-A / PQA-U** abstracts
2. Embeds chunks with **BioBERT** and indexes them in **FAISS** (checkpointed to Google Drive)
3. Retrieves evidence and generates yes/no/maybe answers with a **4-bit quantized LLM**
4. Adds an **explainability layer** (retrieval transparency, evidence highlighting, source attribution, confidence, faithfulness judge)
5. Evaluates on the **PQA-L test split** (classification accuracy + macro-F1, plus generative EM/F1)

---
### Runtime setup (do this first)
`Runtime → Change runtime type → Hardware accelerator → T4 GPU`, then `Runtime → Run all` (or run cells top to bottom).


## 0. Datasets you need in the Colab `Dataset/` folder

Create a folder in your Google Drive (the notebook will mount Drive at `/content/drive`) with this structure:

```
MyDrive/
└── clinical_rag/
    ├── Dataset/
    │   ├── ori_pqal.json        # PQA-L labeled (1000)  -> from the official repo ./data/
    │   ├── ori_pqaa.json        # PQA-A artificial (~211k) -> Google Drive link in repo README
    │   └── ori_pqau.json        # PQA-U unlabeled (~61k, optional corpus) -> repo README link
    └── artifacts/               # notebook writes the FAISS index + embeddings here
```

**Where each file comes from (official repo: github.com/pubmedqa/pubmedqa):**
- `ori_pqal.json` — already in the repo under `./data/`. This is your **evaluation gold standard**.
- `ori_pqaa.json` — download from the Google Drive link in the repo README (large, ~211k). Used only as **corpus / optional fine-tuning fuel**, never as eval ground truth.
- `ori_pqau.json` — optional. Extra unlabeled abstracts to enrich the retrieval corpus.

> **Design decision:** PQA-A/PQA-U are used only to build the retrieval corpus. All reported metrics come from the PQA-L test split. The corpus is deduplicated by PMID so that PQA-L test abstracts do not sit in the corpus in a way that would trivially leak the answer.

If you don't have the PQA-A download yet, the notebook has a fallback that builds the corpus from PQA-L contexts alone so you can still run the full pipeline end-to-end.


## 1. Environment check & install

In [1]:
# Confirm the GPU is a T4 (or similar). If this errors, enable GPU in Runtime settings.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU detected — enable T4 in Runtime settings!")

Tesla T4, 15360 MiB



In [2]:
# Installs. Pinned to a current transformers so Phi-3 loads natively (avoids rope_scaling KeyError).
# bitsandbytes -> 4-bit quantization; faiss-cpu for the vector index.
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes faiss-cpu sentence-transformers 2>/dev/null
print("Installs done.")
print("IMPORTANT: now do Runtime -> Restart session, then continue from Section 2 (do NOT re-run this cell).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.1 MB/s eta 0:00:00
Installs done.
IMPORTANT: now do Runtime -> Restart session, then continue from Section 2 (do NOT re-run this cell).


## 2. Mount Drive & set paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/clinical_rag"
DATA_DIR = os.path.join(BASE, "Dataset")
ART_DIR  = os.path.join(BASE, "artifacts")
os.makedirs(ART_DIR, exist_ok=True)

PQAL = os.path.join(DATA_DIR, "ori_pqal.json")   # required (evaluation)
PQAA = os.path.join(DATA_DIR, "ori_pqaa.json")   # optional (corpus)
PQAU = os.path.join(DATA_DIR, "ori_pqau.json")   # optional (corpus)

print("PQA-L present:", os.path.exists(PQAL))
print("PQA-A present:", os.path.exists(PQAA))
print("PQA-U present:", os.path.exists(PQAU))
assert os.path.exists(PQAL), "ori_pqal.json is required. Put it in Dataset/ (it's in the repo ./data/)."


Mounted at /content/drive
PQA-L present: True
PQA-A present: False
PQA-U present: False


## 3. Config setup

Tune `CORPUS_MAX_ABSTRACTS` down if you hit session limits. 20k is a safe start; 50k is fine if you checkpoint.


In [4]:
CONFIG = {
    # --- corpus ---
    "CORPUS_MAX_ABSTRACTS": 20000,   # subsample of PQA-A/U. Lower = faster. .
    "CHUNK_WORDS": 120,
    "CHUNK_OVERLAP": 30,
    # --- embeddings ---
    "EMBED_MODEL": "dmis-lab/biobert-v1.1",
    "EMBED_BATCH": 64,
    "EMBED_POOLING": "mean",         # mean pooling > CLS for BioBERT retrieval
    "MAX_TOKENS": 256,
    "CHECKPOINT_EVERY": 5000,        # save partial embeddings every N chunks
    # --- retrieval ---
    "TOP_K": 5,
    # --- generation ---
    "LLM_MODEL": "microsoft/Phi-3-mini-4k-instruct",  # fits T4 comfortably; swap for Mistral 7B 4-bit if desired
    "LOAD_IN_4BIT": True,
    "MAX_NEW_TOKENS": 256,
    # --- evaluation ---
    "EVAL_N": 500,                   # PQA-L test split size (official). Lower for a quick smoke test.
    "SEED": 42,
}
import random, numpy as np
random.seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"])
CONFIG

{'CORPUS_MAX_ABSTRACTS': 20000,
 'CHUNK_WORDS': 120,
 'CHUNK_OVERLAP': 30,
 'EMBED_MODEL': 'dmis-lab/biobert-v1.1',
 'EMBED_BATCH': 64,
 'EMBED_POOLING': 'mean',
 'MAX_TOKENS': 256,
 'CHECKPOINT_EVERY': 5000,
 'TOP_K': 5,
 'LLM_MODEL': 'microsoft/Phi-3-mini-4k-instruct',
 'LOAD_IN_4BIT': True,
 'MAX_NEW_TOKENS': 256,
 'EVAL_N': 500,
 'SEED': 42}

## 4. Load data & build the PQA-L test split




In [5]:
import json

def load_json(path):
    with open(path) as f:
        return json.load(f)

pqal = load_json(PQAL)   # dict keyed by PMID
pmids = sorted(pqal.keys())
print(f"PQA-L instances: {len(pmids)}")

# Reproducible test split of size EVAL_N.
rng = random.Random(CONFIG["SEED"])
shuffled = pmids[:]
rng.shuffle(shuffled)
test_pmids = set(shuffled[:CONFIG["EVAL_N"]])
train_pmids = set(shuffled[CONFIG["EVAL_N"]:])
print(f"Test: {len(test_pmids)} | Remaining (dev/corpus): {len(train_pmids)}")

# Quick look at one instance's structure
example = pqal[pmids[0]]
print("\nFields:", list(example.keys()))
print("QUESTION:", example["QUESTION"][:120])
print("final_decision:", example["final_decision"])


PQA-L instances: 1000
Test: 500 | Remaining (dev/corpus): 500

Fields: ['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER']
QUESTION: Is oral endotracheal intubation efficacy impaired in the helicopter environment?
final_decision: yes


## 5. Build the retrieval corpus

Priority order:
1. If PQA-A / PQA-U exist → subsample abstracts from them.
2. Always also add PQA-L **non-test** contexts (train split) so the corpus is rich.
3. **Deduplicate by PMID** and **exclude PQA-L test PMIDs** from the corpus to avoid answer leakage.

Fallback: if neither PQA-A nor PQA-U is present, the corpus is built from PQA-L train contexts alone enough to run the whole pipeline, just smaller.


In [6]:
def abstract_text_from_contexts(rec):
    # PubMedQA 'CONTEXTS' is a list of section strings; join into one abstract.
    ctx = rec.get("CONTEXTS") or []
    return " ".join(ctx).strip()

corpus_records = {}  # pmid -> {"pmid","text","source"}

# 1) ALL PQA-L abstracts (train AND test) form the core retrievable corpus.
#    This is the standard PubMedQA retrieval setup: the task is to retrieve the
#    correct source abstract out of the whole pool. Including test abstracts is
#    NOT leakage -- we retrieve the *context*, never the yes/no/maybe label, so
#    the model still has to reason to an answer. Without this, retrieval hit rate
#    is structurally 0 because the correct abstract would not exist in the corpus.
added_l = 0
for pid in pqal.keys():
    txt = abstract_text_from_contexts(pqal[pid])
    if len(txt) >= 50:
        tag = "PQA-L-test" if pid in test_pmids else "PQA-L-train"
        corpus_records[pid] = {"pmid": pid, "text": txt, "source": tag}
        added_l += 1
print(f"PQA-L (all): added {added_l} abstracts (core retrievable corpus)")

# 2) PQA-A / PQA-U as ADDITIONAL distractor corpus (makes retrieval harder/realistic).
#    These enlarge the pool the correct abstract must be found within. Subsampled
#    to keep the total near CORPUS_MAX_ABSTRACTS for T4 feasibility.
for path, tag in [(PQAA, "PQA-A"), (PQAU, "PQA-U")]:
    if os.path.exists(path):
        data = load_json(path)
        keys = list(data.keys())
        rng.shuffle(keys)
        added = 0
        for pid in keys:
            if pid in corpus_records:      # don't duplicate PQA-L PMIDs
                continue
            txt = abstract_text_from_contexts(data[pid])
            if len(txt) < 50:
                continue
            corpus_records[pid] = {"pmid": pid, "text": txt, "source": tag}
            added += 1
            if len(corpus_records) >= CONFIG["CORPUS_MAX_ABSTRACTS"]:
                break
        print(f"{tag}: added {added} distractor abstracts (corpus now {len(corpus_records)})")
        if len(corpus_records) >= CONFIG["CORPUS_MAX_ABSTRACTS"]:
            break

print(f"\nTOTAL corpus abstracts: {len(corpus_records)}")
n_test_in_corpus = sum(1 for pid in test_pmids if pid in corpus_records)
print(f"Test abstracts present in corpus: {n_test_in_corpus}/{len(test_pmids)} (should be {len(test_pmids)})")
assert len(corpus_records) > 0, "Corpus is empty -- check your Dataset files."
assert n_test_in_corpus == len(test_pmids), "Some test abstracts missing from corpus -- hit rate would be capped below 1."

PQA-L (all): added 1000 abstracts (core retrievable corpus)

TOTAL corpus abstracts: 1000
Test abstracts present in corpus: 500/500 (should be 500)


## 6. Chunking (120 words, 30-word overlap)

Many PubMedQA abstracts are shorter than 120 words, so chunking is often a no-op  that's expected
and fine. Each chunk keeps a back-pointer to its source PMID for source attribution later.


In [7]:
def chunk_words(text, size, overlap):
    words = text.split()
    if len(words) <= size:
        return [text]
    chunks, start = [], 0
    step = max(1, size - overlap)
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
        start += step
    return chunks

chunks = []   # list of dicts: {chunk_id, pmid, source, text}
cid = 0
for pid, rec in corpus_records.items():
    for ch in chunk_words(rec["text"], CONFIG["CHUNK_WORDS"], CONFIG["CHUNK_OVERLAP"]):
        chunks.append({"chunk_id": cid, "pmid": pid, "source": rec["source"], "text": ch})
        cid += 1

print(f"Total chunks: {len(chunks)}")
import numpy as np
lens = np.array([len(c['text'].split()) for c in chunks])
print(f"Chunk word-length: mean {lens.mean():.0f}, median {np.median(lens):.0f}, max {lens.max()}")


Total chunks: 2385
Chunk word-length: mean 101, median 120, max 120


## 7. BioBERT embedding with checkpointing to Drive

BioBERT has no sentence-similarity objective, so we use **mean pooling** over token embeddings
(better than raw CLS for retrieval) and **L2-normalize** so inner product = cosine similarity.

Embeddings are checkpointed to Drive every `CHECKPOINT_EVERY` chunks — if the session drops,
re-run this cell and it resumes from the last checkpoint.


In [8]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["EMBED_MODEL"])
emb_model = AutoModel.from_pretrained(CONFIG["EMBED_MODEL"]).to(device).eval()

def mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return summed / counts

@torch.no_grad()
def embed_texts(texts):
    enc = tok(texts, padding=True, truncation=True,
              max_length=CONFIG["MAX_TOKENS"], return_tensors="pt").to(device)
    out = emb_model(**enc).last_hidden_state
    vecs = mean_pool(out, enc["attention_mask"])
    vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
    return vecs.cpu().numpy().astype("float32")

EMB_PATH = os.path.join(ART_DIR, "chunk_embeddings.npy")
META_PATH = os.path.join(ART_DIR, "chunk_meta.json")

# Resume logic
start_idx = 0
if os.path.exists(EMB_PATH) and os.path.exists(META_PATH):
    existing = np.load(EMB_PATH)
    start_idx = existing.shape[0]
    print(f"Resuming from checkpoint: {start_idx} chunks already embedded.")
    all_vecs = [existing]
else:
    all_vecs = []

B = CONFIG["EMBED_BATCH"]
buffer = []
from time import time
t0 = time()
for i in range(start_idx, len(chunks), B):
    batch = [c["text"] for c in chunks[i:i+B]]
    buffer.append(embed_texts(batch))
    done = i + len(batch)
    if done % CONFIG["CHECKPOINT_EVERY"] < B or done >= len(chunks):
        current = np.concatenate(all_vecs + buffer, axis=0) if (all_vecs or buffer) else np.zeros((0,768),dtype="float32")
        np.save(EMB_PATH, current)
        with open(META_PATH, "w") as f:
            json.dump(chunks[:current.shape[0]], f)
        all_vecs = [current]; buffer = []
        print(f"  checkpoint @ {done}/{len(chunks)} chunks  ({time()-t0:.0f}s)")

embeddings = np.load(EMB_PATH)
print(f"\nEmbeddings ready: {embeddings.shape}")


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Resuming from checkpoint: 2385 chunks already embedded.

Embeddings ready: (2385, 768)


## 8. FAISS index (IndexFlatIP = exact cosine on normalized vectors)



In [9]:
import faiss
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)
faiss.write_index(index, os.path.join(ART_DIR, "faiss.index"))
print("FAISS index size:", index.ntotal)

# Reload metadata aligned to the index
with open(META_PATH) as f:
    chunk_meta = json.load(f)
assert len(chunk_meta) == index.ntotal, "Meta/index length mismatch — re-run embedding cell."


FAISS index size: 2385


## 9. Retrieval function

In [10]:
def retrieve(question, top_k=None):
    top_k = top_k or CONFIG["TOP_K"]
    qv = embed_texts([question])          # already normalized
    scores, idxs = index.search(qv, top_k)
    hits = []
    for rank, (s, ix) in enumerate(zip(scores[0], idxs[0]), start=1):
        m = chunk_meta[ix]
        hits.append({
            "rank": rank,
            "similarity": float(s),
            "chunk_id": m["chunk_id"],
            "pmid": m["pmid"],
            "source": m["source"],
            "text": m["text"],
        })
    return hits

# smoke test
demo = retrieve("Does aspirin reduce the risk of heart attack?")
for h in demo[:3]:
    print(f"rank {h['rank']}  sim {h['similarity']:.3f}  pmid {h['pmid']}  {h['text'][:90]}...")


rank 1  sim 0.897  pmid 11340218  In primary and secondary prevention trials, statins have been shown to reduce the risk of ...
rank 2  sim 0.895  pmid 15208005  Low intakes or blood levels of eicosapentaenoic and docosahexaenoic acids (EPA + DHA) are ...
rank 3  sim 0.893  pmid 10732884  no impact on in-hospital mortality in patients with a history of MI. Furthermore, coronary...


## 10. Load the generator LLM (4-bit)



In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
llm_tok = AutoTokenizer.from_pretrained(CONFIG["LLM_MODEL"])
llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["LLM_MODEL"],
    quantization_config=bnb if CONFIG["LOAD_IN_4BIT"] else None,
    device_map="auto",
    torch_dtype=torch.float16,
)
llm.eval()
print("LLM loaded:", CONFIG["LLM_MODEL"])
print("GPU mem allocated: %.1f GB" % (torch.cuda.memory_allocated()/1e9))


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

LLM loaded: microsoft/Phi-3-mini-4k-instruct
GPU mem allocated: 2.7 GB


In [12]:
@torch.no_grad()
def llm_generate(prompt, max_new_tokens=None):
    max_new_tokens = max_new_tokens or CONFIG["MAX_NEW_TOKENS"]
    messages = [{"role": "user", "content": prompt}]
    # Render the chat template to a string, then tokenize -> gives input_ids + attention_mask.
    text = llm_tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = llm_tok(text, return_tensors="pt").to(llm.device)
    input_len = enc["input_ids"].shape[1]
    out = llm.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=llm_tok.eos_token_id,
    )
    return llm_tok.decode(out[0][input_len:], skip_special_tokens=True).strip()


## 11. Prompt construction & grounded answering



In [13]:
import re

def build_prompt(question, hits):
    evidence = "\n\n".join(
        f"[Source {h['rank']} | PMID {h['pmid']} | chunk {h['chunk_id']}]\n{h['text']}"
        for h in hits
    )
    return f"""You are a biomedical question-answering assistant. Answer ONLY using the retrieved evidence below. Do not use outside knowledge. Do not fabricate references.

RETRIEVED EVIDENCE:
{evidence}

QUESTION: {question}

Decide whether the evidence answers the question as yes, no, or maybe. Use 'maybe' ONLY when the evidence is genuinely mixed or insufficient; prefer a definite yes or no when the evidence supports one.

Respond in EXACTLY this format and nothing else:
ANSWER: <2-3 sentence evidence-based answer>
DECISION: <yes|no|maybe>
CONFIDENCE: <one sentence>"""

def parse_decision(text):
    # Prefer the explicit DECISION: line. If several, take the last one.
    matches = re.findall(r"DECISION:\s*\**\s*(yes|no|maybe)", text, re.IGNORECASE)
    if matches:
        return matches[-1].lower()
    # Fallback: last bare yes/no/maybe token anywhere (last is usually the conclusion).
    bare = re.findall(r"\b(yes|no|maybe)\b", text, re.IGNORECASE)
    return bare[-1].lower() if bare else "maybe"

def answer_question(question, top_k=None):
    hits = retrieve(question, top_k)
    prompt = build_prompt(question, hits)
    raw = llm_generate(prompt)
    return {"question": question, "hits": hits, "raw": raw, "decision": parse_decision(raw)}

# demo
# r = answer_question(pqal[list(test_pmids)[0]]["QUESTION"])
# print(r["raw"])
# print("\nParsed decision:", r["decision"])

## 12. Explainability layer




In [14]:
def similarity_to_confidence(sim):
    if sim >= 0.90: return "High"
    if sim >= 0.80: return "Medium"
    return "Low"

def highlight_evidence(answer_text, hits, max_sentences=3):
    # Simple lexical overlap highlighter: rank evidence sentences by word overlap with the answer.
    ans_words = set(re.findall(r"[a-z]{4,}", answer_text.lower()))
    scored = []
    for h in hits:
        for sent in re.split(r"(?<=[.!?])\s+", h["text"]):
            sw = set(re.findall(r"[a-z]{4,}", sent.lower()))
            ov = len(ans_words & sw)
            if ov > 0:
                scored.append((ov, h["pmid"], sent.strip()))
    scored.sort(reverse=True)
    return scored[:max_sentences]

@torch.no_grad()
def faithfulness_judge(answer_text, hits):
    evidence = "\n".join(f"- {h['text']}" for h in hits)
    judge_prompt = f"""You are a strict fact-checker. Determine if the ANSWER is fully supported by the EVIDENCE.

EVIDENCE:
{evidence}

ANSWER:
{answer_text}

Reply with exactly one word: SUPPORTED or NOT_SUPPORTED."""
    verdict = llm_generate(judge_prompt, max_new_tokens=10).upper()
    return "SUPPORTED" if "NOT_SUPPORTED" not in verdict and "SUPPORT" in verdict else "NOT_SUPPORTED"

def explain(result):
    print("="*70)
    print("QUESTION:", result["question"])
    print("\n--- 1. RETRIEVAL TRANSPARENCY ---")
    print(f"{'Rank':<5}{'Similarity':<12}{'PMID':<12}{'Confidence'}")
    for h in result["hits"]:
        print(f"{h['rank']:<5}{h['similarity']:<12.3f}{h['pmid']:<12}{similarity_to_confidence(h['similarity'])}")
    print("\n--- 2. GENERATED ANSWER ---")
    print(result["raw"])
    print("\n--- 3. EVIDENCE HIGHLIGHTING (supporting sentences) ---")
    for ov, pmid, sent in highlight_evidence(result["raw"], result["hits"]):
        print(f"  [PMID {pmid} | overlap {ov}] {sent[:140]}")
    print("\n--- 4. SOURCE ATTRIBUTION ---")
    for h in result["hits"]:
        print(f"  Source {h['rank']}: PMID {h['pmid']} (chunk {h['chunk_id']}, {h['source']})")
    print("\n--- 5. FAITHFULNESS ---")
    print("  Verdict:", faithfulness_judge(result["raw"], result["hits"]))
    print("="*70)

# explain(r)   # demo call disabled for baseline-only run (uncomment to see one example)


## 12b. Quick sanity check BEFORE the full evaluation


1. **Retrieval works** — the correct PMID should now appear in the top-k for most questions (hit rate > 0).
2. **Decisions parse** — the model should emit a clear DECISION and predictions should NOT all be "maybe".


In [15]:
# ==== SECTION 12b sanity check DISABLED (already validated) ====
# Uncomment to re-run the 10-question retrieval/parse sanity check.
"""
from collections import Counter as _C
sample_pmids = list(test_pmids)[:10]
hits_ok, preds = 0, []
for k, pid in enumerate(sample_pmids):
    res = answer_question(pqal[pid]["QUESTION"])
    got_pmids = [h["pmid"] for h in res["hits"]]
    hit = pid in got_pmids
    hits_ok += hit
    preds.append(res["decision"])
    if k < 3:  # show first 3 full outputs for inspection
        print(f"--- Q{k+1} (gold={pqal[pid]['final_decision']}, pred={res['decision']}, retrieval_hit={hit}) ---")
        print(res["raw"][:400])
        print(f"retrieved PMIDs: {got_pmids}  | correct PMID: {pid}")
        print()

print("="*50)
print(f"Sample retrieval hit rate: {hits_ok}/{len(sample_pmids)} = {hits_ok/len(sample_pmids):.2f}")
print(f"Sample prediction distribution: {dict(_C(preds))}")
print("Expect: hit rate well above 0, and a MIX of yes/no/maybe (not all maybe).")
"""

'\nfrom collections import Counter as _C\nsample_pmids = list(test_pmids)[:10]\nhits_ok, preds = 0, []\nfor k, pid in enumerate(sample_pmids):\n    res = answer_question(pqal[pid]["QUESTION"])\n    got_pmids = [h["pmid"] for h in res["hits"]]\n    hit = pid in got_pmids\n    hits_ok += hit\n    preds.append(res["decision"])\n    if k < 3:  # show first 3 full outputs for inspection\n        print(f"--- Q{k+1} (gold={pqal[pid][\'final_decision\']}, pred={res[\'decision\']}, retrieval_hit={hit}) ---")\n        print(res["raw"][:400])\n        print(f"retrieved PMIDs: {got_pmids}  | correct PMID: {pid}")\n        print()\n\nprint("="*50)\nprint(f"Sample retrieval hit rate: {hits_ok}/{len(sample_pmids)} = {hits_ok/len(sample_pmids):.2f}")\nprint(f"Sample prediction distribution: {dict(_C(preds))}")\nprint("Expect: hit rate well above 0, and a MIX of yes/no/maybe (not all maybe).")\n'

## 13. Evaluation on the PQA-L test split

Primary metric: **classification accuracy + macro-F1** over yes/no/maybe (the native PubMedQA task).
Secondary: **retrieval hit rate** (did we retrieve a chunk from the correct PMID?) and
**faithfulness rate**. Run over `EVAL_N` instances — checkpoints so a disconnect won't lose progress.


In [16]:
from collections import Counter
import json as _json

RESULTS_PATH = os.path.join(ART_DIR, "eval_results.json")

# resume
done_results = {}
if os.path.exists(RESULTS_PATH):
    done_results = load_json(RESULTS_PATH)
    print(f"Resuming — {len(done_results)} already evaluated.")

# ==== EVAL LOOP DISABLED FOR BASELINE-ONLY RUN ====
# Your eval_results.json already contains the full 500-question BioBERT run.
# To RE-RUN the main evaluation from scratch, delete eval_results.json and
# uncomment the block below (remove the leading # on each line).
# test_list = sorted(test_pmids)
# for i, pid in enumerate(test_list):
#     if pid in done_results:
#         continue
#     rec = pqal[pid]
#     res = answer_question(rec["QUESTION"])
#     hit = any(h["pmid"] == pid for h in res["hits"])   # retrieval hit rate
#     faith = faithfulness_judge(res["raw"], res["hits"])
#     done_results[pid] = {
#         "pred": res["decision"],
#         "gold": rec["final_decision"],
#         "retrieval_hit": hit,
#         "faithful": faith,
#     }
#     if (i+1) % 25 == 0:
#         with open(RESULTS_PATH, "w") as f:
#             _json.dump(done_results, f)
#         print(f"  {i+1}/{len(test_list)} evaluated (checkpointed)")

# with open(RESULTS_PATH, "w") as f:
#     _json.dump(done_results, f)
# print("Evaluation complete:", len(done_results))


Resuming — 500 already evaluated.


In [17]:
# Metrics
from collections import defaultdict

labels = ["yes", "no", "maybe"]
gold = [done_results[p]["gold"] for p in done_results]
pred = [done_results[p]["pred"] for p in done_results]

acc = sum(g == p for g, p in zip(gold, pred)) / len(gold)

def macro_f1(gold, pred, labels):
    f1s = []
    for lab in labels:
        tp = sum(g == lab and p == lab for g, p in zip(gold, pred))
        fp = sum(g != lab and p == lab for g, p in zip(gold, pred))
        fn = sum(g == lab and p != lab for g, p in zip(gold, pred))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec  = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        f1s.append(f1)
    return sum(f1s)/len(f1s), dict(zip(labels, f1s))

mf1, per_class = macro_f1(gold, pred, labels)
hit_rate = sum(done_results[p]["retrieval_hit"] for p in done_results)/len(done_results)
faith_rate = sum(done_results[p]["faithful"]=="SUPPORTED" for p in done_results)/len(done_results)

print(f"PQA-L test instances : {len(gold)}")
print(f"Accuracy             : {acc:.3f}")
print(f"Macro-F1             : {mf1:.3f}")
print(f"  per-class F1       : {per_class}")
print(f"Retrieval hit rate   : {hit_rate:.3f}")
print(f"Faithfulness (SUPPORTED) rate: {faith_rate:.3f}")
print(f"Hallucination rate   : {1-faith_rate:.3f}")
print("\nGold distribution :", dict(Counter(gold)))
print("Pred distribution :", dict(Counter(pred)))


PQA-L test instances : 500
Accuracy             : 0.532
Macro-F1             : 0.472
  per-class F1       : {'yes': 0.66015625, 'no': 0.5071428571428572, 'maybe': 0.25}
Retrieval hit rate   : 0.880
Faithfulness (SUPPORTED) rate: 0.750
Hallucination rate   : 0.250

Gold distribution : {'yes': 273, 'maybe': 63, 'no': 164}
Pred distribution : {'yes': 239, 'maybe': 145, 'no': 116}


---
## 16. Baseline experiments (for the comparison table)

Two controls that isolate what your design choices contribute:
- **No-RAG:** the same LLM answers with NO retrieved evidence -> shows whether retrieval helps at all.
- **MiniLM-RAG:** the same pipeline but a generic embedder (all-MiniLM-L6-v2) instead of BioBERT -> shows whether the biomedical embedder specifically helps.

Each writes its own results file so your main BioBERT run (eval_results.json) is untouched. These add runtime (each is another pass over the 500 test questions); run them when you have GPU budget. Set EVAL_N smaller first to smoke-test.

In [18]:
import json as _json, os

# Shared metric helper (reuses the same logic as Section 13)
def compute_metrics(results):
    labs = ["yes", "no", "maybe"]
    g = [results[p]["gold"] for p in results]
    p_ = [results[p]["pred"] for p in results]
    acc = sum(a == b for a, b in zip(g, p_)) / len(g)
    f1s = []
    for lab in labs:
        tp = sum(a == lab and b == lab for a, b in zip(g, p_))
        fp = sum(a != lab and b == lab for a, b in zip(g, p_))
        fn = sum(a == lab and b != lab for a, b in zip(g, p_))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    mf1 = sum(f1s) / len(f1s)
    hit = sum(results[p].get("retrieval_hit", False) for p in results) / len(results)
    faith = sum(results[p].get("faithful") == "SUPPORTED" for p in results) / len(results)
    return {"n": len(g), "accuracy": acc, "macro_f1": mf1,
            "per_class_f1": dict(zip(labs, f1s)),
            "retrieval_hit_rate": hit, "faithful_rate": faith}


# ---- BASELINE 1: No-RAG (question only, no retrieved evidence) ----
def build_prompt_norag(question):
    return f"""You are a biomedical question-answering assistant. Answer the question using your own knowledge.

QUESTION: {question}

Decide whether the answer is yes, no, or maybe. Use 'maybe' ONLY when genuinely uncertain.

Respond in EXACTLY this format and nothing else:
ANSWER: <2-3 sentence answer>
DECISION: <yes|no|maybe>
CONFIDENCE: <one sentence>"""

def answer_norag(question):
    raw = llm_generate(build_prompt_norag(question))
    return {"raw": raw, "decision": parse_decision(raw)}

NORAG_PATH = os.path.join(ART_DIR, "eval_norag.json")
norag = load_json(NORAG_PATH) if os.path.exists(NORAG_PATH) else {}
test_list = sorted(test_pmids)
for i, pid in enumerate(test_list):
    if pid in norag:
        continue
    rec = pqal[pid]
    res = answer_norag(rec["QUESTION"])
    norag[pid] = {"pred": res["decision"], "gold": rec["final_decision"],
                  "retrieval_hit": False, "faithful": "N/A"}  # no retrieval in this baseline
    if (i + 1) % 50 == 0:
        with open(NORAG_PATH, "w") as f: _json.dump(norag, f)
        print(f"  [no-RAG] {i+1}/{len(test_list)}")
with open(NORAG_PATH, "w") as f: _json.dump(norag, f)
print("No-RAG baseline done.")

No-RAG baseline done.


In [19]:
# ---- BASELINE 2: MiniLM-RAG (same pipeline, generic embedder instead of BioBERT) ----
# Re-embeds the SAME corpus chunks with all-MiniLM-L6-v2 and builds a separate FAISS index,
# so the only variable that changes vs your main system is the embedding model.
from sentence_transformers import SentenceTransformer
import numpy as np, faiss, os, json as _json

minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

def embed_texts_minilm(texts):
    v = minilm.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                      show_progress_bar=False)
    return v.astype("float32")

MINILM_EMB = os.path.join(ART_DIR, "minilm_embeddings.npy")
if os.path.exists(MINILM_EMB):
    minilm_emb = np.load(MINILM_EMB)
    print("Loaded cached MiniLM embeddings:", minilm_emb.shape)
else:
    vecs = []
    B = 128
    for i in range(0, len(chunk_meta), B):
        batch = [chunk_meta[j]["text"] for j in range(i, min(i+B, len(chunk_meta)))]
        vecs.append(embed_texts_minilm(batch))
        if (i // B) % 20 == 0:
            print(f"  [MiniLM embed] {min(i+B, len(chunk_meta))}/{len(chunk_meta)}")
    minilm_emb = np.concatenate(vecs, axis=0)
    np.save(MINILM_EMB, minilm_emb)
    print("MiniLM embeddings built:", minilm_emb.shape)

minilm_index = faiss.IndexFlatIP(minilm_emb.shape[1])
minilm_index.add(minilm_emb)

def retrieve_minilm(question, top_k=None):
    top_k = top_k or CONFIG["TOP_K"]
    qv = embed_texts_minilm([question])
    scores, idxs = minilm_index.search(qv, top_k)
    hits = []
    for rank, (s, ix) in enumerate(zip(scores[0], idxs[0]), start=1):
        m = chunk_meta[ix]
        hits.append({"rank": rank, "similarity": float(s), "chunk_id": m["chunk_id"],
                     "pmid": m["pmid"], "source": m["source"], "text": m["text"]})
    return hits

def answer_minilm(question, top_k=None):
    hits = retrieve_minilm(question, top_k)
    raw = llm_generate(build_prompt(question, hits))
    return {"hits": hits, "raw": raw, "decision": parse_decision(raw)}

MINILM_PATH = os.path.join(ART_DIR, "eval_minilm.json")
minilm_res = load_json(MINILM_PATH) if os.path.exists(MINILM_PATH) else {}
for i, pid in enumerate(sorted(test_pmids)):
    if pid in minilm_res:
        continue
    rec = pqal[pid]
    res = answer_minilm(rec["QUESTION"])
    hit = any(h["pmid"] == pid for h in res["hits"])
    faith = faithfulness_judge(res["raw"], res["hits"])
    minilm_res[pid] = {"pred": res["decision"], "gold": rec["final_decision"],
                       "retrieval_hit": hit, "faithful": faith}
    if (i + 1) % 25 == 0:
        with open(MINILM_PATH, "w") as f: _json.dump(minilm_res, f)
        print(f"  [MiniLM-RAG] {i+1}/{len(test_pmids)}")
with open(MINILM_PATH, "w") as f: _json.dump(minilm_res, f)
print("MiniLM-RAG baseline done.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded cached MiniLM embeddings: (2385, 384)
MiniLM-RAG baseline done.


## 16c. Third embedder — S-PubMedBert-RAG (domain + similarity objective)

The decisive comparison. MiniLM has a sentence-similarity objective but general vocabulary; BioBERT has biomedical vocabulary but no similarity objective.  has **both** — biomedical domain plus retrieval fine-tuning. Reuses the same corpus chunks and pipeline; only the embedder changes. Checkpoints to eval_spubmed.json and resumes if interrupted.

In [20]:
# ---- BASELINE 3: S-PubMedBert-RAG (biomedical embedder WITH a sentence-similarity objective) ----
# This is the key comparison: MiniLM has similarity training but general vocab; BioBERT has
# biomedical vocab but no similarity training. S-PubMedBert-MS-MARCO has BOTH -- biomedical
# domain AND a sentence-similarity/retrieval objective (fine-tuned on MS-MARCO). If it wins,
# the story is clean: retrieval quality needs domain knowledge + a similarity objective together.
from sentence_transformers import SentenceTransformer
import numpy as np, faiss, os, json as _json

SPUBMED_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"
spubmed = SentenceTransformer(SPUBMED_NAME, device=device)

def embed_texts_spubmed(texts):
    v = spubmed.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                       show_progress_bar=False)
    return v.astype("float32")

SPUBMED_EMB = os.path.join(ART_DIR, "spubmed_embeddings.npy")
if os.path.exists(SPUBMED_EMB):
    spubmed_emb = np.load(SPUBMED_EMB)
    print("Loaded cached S-PubMedBert embeddings:", spubmed_emb.shape)
else:
    vecs = []
    B = 128
    for i in range(0, len(chunk_meta), B):
        batch = [chunk_meta[j]["text"] for j in range(i, min(i+B, len(chunk_meta)))]
        vecs.append(embed_texts_spubmed(batch))
        if (i // B) % 20 == 0:
            print(f"  [S-PubMedBert embed] {min(i+B, len(chunk_meta))}/{len(chunk_meta)}")
    spubmed_emb = np.concatenate(vecs, axis=0)
    np.save(SPUBMED_EMB, spubmed_emb)
    print("S-PubMedBert embeddings built:", spubmed_emb.shape)

spubmed_index = faiss.IndexFlatIP(spubmed_emb.shape[1])
spubmed_index.add(spubmed_emb)

def retrieve_spubmed(question, top_k=None):
    top_k = top_k or CONFIG["TOP_K"]
    qv = embed_texts_spubmed([question])
    scores, idxs = spubmed_index.search(qv, top_k)
    hits = []
    for rank, (s, ix) in enumerate(zip(scores[0], idxs[0]), start=1):
        m = chunk_meta[ix]
        hits.append({"rank": rank, "similarity": float(s), "chunk_id": m["chunk_id"],
                     "pmid": m["pmid"], "source": m["source"], "text": m["text"]})
    return hits

def answer_spubmed(question, top_k=None):
    hits = retrieve_spubmed(question, top_k)
    raw = llm_generate(build_prompt(question, hits))
    return {"hits": hits, "raw": raw, "decision": parse_decision(raw)}

SPUBMED_PATH = os.path.join(ART_DIR, "eval_spubmed.json")
spubmed_res = load_json(SPUBMED_PATH) if os.path.exists(SPUBMED_PATH) else {}
for i, pid in enumerate(sorted(test_pmids)):
    if pid in spubmed_res:
        continue
    rec = pqal[pid]
    res = answer_spubmed(rec["QUESTION"])
    hit = any(h["pmid"] == pid for h in res["hits"])
    faith = faithfulness_judge(res["raw"], res["hits"])
    spubmed_res[pid] = {"pred": res["decision"], "gold": rec["final_decision"],
                        "retrieval_hit": hit, "faithful": faith}
    if (i + 1) % 25 == 0:
        with open(SPUBMED_PATH, "w") as f: _json.dump(spubmed_res, f)
        print(f"  [S-PubMedBert-RAG] {i+1}/{len(test_pmids)}")
with open(SPUBMED_PATH, "w") as f: _json.dump(spubmed_res, f)
print("S-PubMedBert-RAG baseline done.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  [S-PubMedBert embed] 128/2385
S-PubMedBert embeddings built: (2385, 768)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [S-PubMedBert-RAG] 25/500
  [S-PubMedBert-RAG] 50/500
  [S-PubMedBert-RAG] 75/500
  [S-PubMedBert-RAG] 100/500
  [S-PubMedBert-RAG] 125/500
  [S-PubMedBert-RAG] 150/500
  [S-PubMedBert-RAG] 175/500
  [S-PubMedBert-RAG] 200/500
  [S-PubMedBert-RAG] 225/500
  [S-PubMedBert-RAG] 250/500
  [S-PubMedBert-RAG] 275/500
  [S-PubMedBert-RAG] 300/500
  [S-PubMedBert-RAG] 325/500
  [S-PubMedBert-RAG] 350/500
  [S-PubMedBert-RAG] 375/500
  [S-PubMedBert-RAG] 400/500
  [S-PubMedBert-RAG] 425/500
  [S-PubMedBert-RAG] 450/500
  [S-PubMedBert-RAG] 475/500
  [S-PubMedBert-RAG] 500/500
S-PubMedBert-RAG baseline done.


## 17. Results comparison

Combines all three systems into one table for your dissertation. Run after the baselines complete (it skips any whose results file is missing).

In [21]:
# ---- COMPARISON TABLE: no-RAG vs MiniLM-RAG vs BioBERT-RAG vs S-PubMedBert-RAG ----
import json as _json, os

def load_results(path):
    return load_json(path) if os.path.exists(path) else None

systems = {
    "No-RAG (Phi-3 only)":        load_results(os.path.join(ART_DIR, "eval_norag.json")),
    "MiniLM-RAG":                 load_results(os.path.join(ART_DIR, "eval_minilm.json")),
    "BioBERT-RAG (proposed)":     load_results(RESULTS_PATH),
    "S-PubMedBert-RAG":           load_results(os.path.join(ART_DIR, "eval_spubmed.json")),
}

rows = []
for name, res in systems.items():
    if not res:
        print(f"(skipped {name}: no results file yet)")
        continue
    rows.append((name, compute_metrics(res)))

hdr = f"{'System':<26}{'Acc':>7}{'MacroF1':>9}{'HitRate':>9}{'Faithful':>10}{'Halluc':>8}"
print(hdr)
print("-" * len(hdr))
for name, m in rows:
    norag = (name == "No-RAG (Phi-3 only)")
    hit   = "  n/a" if norag else f"{m['retrieval_hit_rate']:.3f}"
    faith = "  n/a" if norag else f"{m['faithful_rate']:.3f}"
    hal   = "  n/a" if norag else f"{1-m['faithful_rate']:.3f}"
    print(f"{name:<26}{m['accuracy']:>7.3f}{m['macro_f1']:>9.3f}{hit:>9}{faith:>10}{hal:>8}")

print("\nPer-class F1:")
for name, m in rows:
    print(f"  {name:<26} {({k: round(v,3) for k,v in m['per_class_f1'].items()})}")

summary = {name: compute_metrics(res) for name, res in systems.items() if res}
with open(os.path.join(ART_DIR, "comparison_summary.json"), "w") as f:
    _json.dump(summary, f, indent=2)
print("\nSaved comparison_summary.json to artifacts/.")

System                        Acc  MacroF1  HitRate  Faithful  Halluc
---------------------------------------------------------------------
No-RAG (Phi-3 only)         0.322    0.293      n/a       n/a     n/a
MiniLM-RAG                  0.640    0.543    0.996     0.910   0.090
BioBERT-RAG (proposed)      0.532    0.472    0.880     0.750   0.250
S-PubMedBert-RAG            0.652    0.557    0.994     0.906   0.094

Per-class F1:
  No-RAG (Phi-3 only)        {'yes': 0.465, 'no': 0.234, 'maybe': 0.179}
  MiniLM-RAG                 {'yes': 0.762, 'no': 0.644, 'maybe': 0.222}
  BioBERT-RAG (proposed)     {'yes': 0.66, 'no': 0.507, 'maybe': 0.25}
  S-PubMedBert-RAG           {'yes': 0.767, 'no': 0.636, 'maybe': 0.267}

Saved comparison_summary.json to artifacts/.


## 14.  Generative EM / F1 against LONG_ANSWER



In [22]:

# Optional secondary metric, re-runs generation on 100 questions (~10-20 min).


def normalize(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return " ".join(s.split())

def token_f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return 0.0
    common = Counter(p) & Counter(g)
    num = sum(common.values())
    if num == 0:
        return 0.0
    prec, rec = num/len(p), num/len(g)
    return 2*prec*rec/(prec+rec)

# Extract just the ANSWER: line from each raw generation for a fairer comparison.
def extract_answer(raw):
    m = re.search(r"ANSWER:\s*(.+?)(?:DECISION:|$)", raw, re.DOTALL|re.IGNORECASE)
    return m.group(1).strip() if m else raw

# Re-generate answers only if you want text metrics; else skip. Here we reuse stored decisions'
# questions but need the raw text, so this is a light re-run over a small sample to stay in budget.
SAMPLE = list(test_pmids)[:100]   # keep it small on free T4
f1s, ems = [], []
for pid in SAMPLE:
    rec = pqal[pid]
    res = answer_question(rec["QUESTION"])
    ans = extract_answer(res["raw"])
    gold_long = rec.get("LONG_ANSWER", "")
    f1s.append(token_f1(ans, gold_long))
    ems.append(1.0 if normalize(ans) == normalize(gold_long) else 0.0)

print(f"Generative (n={len(SAMPLE)})  token-F1: {np.mean(f1s):.3f}   EM: {np.mean(ems):.3f}")



<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2078/1154290385.py:23: SyntaxWarning: invalid escape sequence '\s'
  m = re.search(r"ANSWER:\s*(.+?)(?:DECISION:|$)", raw, re.DOTALL|re.IGNORECASE)


'\ndef normalize(s):\n    s = s.lower()\n    s = re.sub(r"[^a-z0-9 ]", " ", s)\n    return " ".join(s.split())\n\ndef token_f1(pred, gold):\n    p, g = normalize(pred).split(), normalize(gold).split()\n    if not p or not g:\n        return 0.0\n    common = Counter(p) & Counter(g)\n    num = sum(common.values())\n    if num == 0:\n        return 0.0\n    prec, rec = num/len(p), num/len(g)\n    return 2*prec*rec/(prec+rec)\n\n# Extract just the ANSWER: line from each raw generation for a fairer comparison.\ndef extract_answer(raw):\n    m = re.search(r"ANSWER:\\s*(.+?)(?:DECISION:|$)", raw, re.DOTALL|re.IGNORECASE)\n    return m.group(1).strip() if m else raw\n\n# Re-generate answers only if you want text metrics; else skip. Here we reuse stored decisions\'\n# questions but need the raw text, so this is a light re-run over a small sample to stay in budget.\nSAMPLE = list(test_pmids)[:100]   # keep it small on free T4\nf1s, ems = [], []\nfor pid in SAMPLE:\n    rec = pqal[pid]\n    res 

## 15. Interactive demo — ask your own clinical question

In [23]:
def ask(question):
    res = answer_question(question)
    explain(res)
    return res

# _ = ask("Is metformin effective as a first-line treatment for type 2 diabetes?")
